# 02 — Обучение Uzbek NER на основе baseline

Ноутбук реализует полный воспроизводимый цикл для конкурсной задачи:

1. проверка схемы, контрольных сумм и символьных границ;
2. преобразование exact character spans в BIO-разметку;
3. токенизация **исходного текста без изменяющей длину нормализации**;
4. sliding windows для длинных документов;
5. fine-tuning multilingual Transformer;
6. выбор checkpoint по **exact-span micro-F1**, а не по token accuracy или loss;
7. Precision / Recall / F1 по ORG, NAME, GEO, micro и macro;
8. сохранение модели, предсказаний, метрик и примеров ошибок.

Модель и параметры окон по умолчанию берутся из \`baseline/common.py\`. Для быстрого smoke-test измените поле \`fast_run\` в \`Config\` на \`True\`.

> Важное правило: апострофы, Unicode, регистр и пробелы в исходной строке не меняются. Иначе координаты \`start/end\` перестанут соответствовать конкурсному gold.


## 0. Окружение

Зависимости проекта уже перечислены в \`ner_uz_hackathon_participant/requirements.txt\`. При необходимости выполните следующую ячейку один раз, затем перезапустите kernel. CUDA-сборку PyTorch устанавливайте только в совместимом Linux/CUDA-окружении.


In [1]:
# Раскомментируйте подходящую команду при необходимости.
%pip install -r ner_uz_hackathon_participant/requirements.txt
# Универсальный вариант для notebook-окружения без CUDA-specific index:
%pip install "torch>=2.3" "transformers>=4.45" "tokenizers>=0.20" "tqdm>=4.66"


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
ERROR: Could not find a version that satisfies the requirement torch==2.6.0+cu124 (from versions: 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0, 2.14.0)
ERROR: No matching distribution found for torch==2.6.0+cu124
Note: you may need to restart the kernel to use updated packages.
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata

## 1. Конфигурация и воспроизводимость

Пути определяются автоматически: ноутбук можно запускать из корня репозитория или из каталога участника.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import sys
import time
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import torch
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    get_linear_schedule_with_warmup,
)

def find_project_dir() -> Path:
    here = Path.cwd().resolve()
    candidates = [
        here,
        here / "ner_uz_hackathon_participant",
        here.parent / "ner_uz_hackathon_participant",
    ]
    for candidate in candidates:
        if (candidate / "data/train.jsonl").exists() and (candidate / "scripts/evaluate.py").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден каталог ner_uz_hackathon_participant с data/train.jsonl. "
        "Запустите notebook из корня репозитория."
    )

PROJECT_DIR = find_project_dir()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Единая реализация подготовки, обучения и инференса из папки baseline.
from baseline.common import (
    TokenizedNerDataset as BaselineTokenizedNerDataset,
    load_fast_tokenizer as baseline_load_fast_tokenizer,
    validate_window as baseline_validate_window,
)
from baseline.predict import (
    _build_windows as baseline_build_windows,
    _decode_records as baseline_decode_records,
    _predict_token_scores as baseline_predict_token_scores,
)
from baseline.train import train_epoch as baseline_train_epoch
TRAIN_PATH = PROJECT_DIR / "data/train.jsonl"
DEV_PATH = PROJECT_DIR / "data/dev.jsonl"
MANIFEST_PATH = PROJECT_DIR / "data/dataset_manifest.json"
ARTIFACT_DIR = PROJECT_DIR / "artifacts/baseline_notebook"
MODEL_DIR = ARTIFACT_DIR / "model"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "FacebookAI/xlm-roberta-large"
MAX_LENGTH = 512
STRIDE = 128

@dataclass(frozen=True)
class Config:
    model_name: str = MODEL
    max_length: int = MAX_LENGTH
    stride: int = STRIDE
    epochs: int = 4
    train_batch_size: int = 8
    eval_batch_size: int = 16
    gradient_accumulation_steps: int = 2
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.10
    max_grad_norm: float = 1.0
    seed: int = 42
    num_workers: int = 0
    use_amp: bool = True
    fast_run: bool = False
    fast_train_records: int = 256
    fast_dev_records: int = 128

CFG = Config()

def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(CFG.seed)
DEVICE = resolve_device()
USE_AMP = CFG.use_amp and DEVICE.type == "cuda"

print("Project:", PROJECT_DIR)
print("Artifacts:", ARTIFACT_DIR)
print("Device:", DEVICE, "| AMP:", USE_AMP)
print(json.dumps(asdict(CFG), ensure_ascii=False, indent=2))


/Users/dashakoryak/Desktop/ITMO/hackaton/multiscript-uzbek-ner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: /Users/dashakoryak/Desktop/ITMO/hackaton/multiscript-uzbek-ner/ner_uz_hackathon_participant
Artifacts: /Users/dashakoryak/Desktop/ITMO/hackaton/multiscript-uzbek-ner/ner_uz_hackathon_participant/artifacts/baseline_notebook
Device: mps | AMP: False
{
  "model_name": "FacebookAI/xlm-roberta-large",
  "max_length": 512,
  "stride": 128,
  "epochs": 4,
  "train_batch_size": 8,
  "eval_batch_size": 16,
  "gradient_accumulation_steps": 2,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "max_grad_norm": 1.0,
  "seed": 42,
  "num_workers": 0,
  "use_amp": true,
  "fast_run": false,
  "fast_train_records": 256,
  "fast_dev_records": 128
}


## 2. Загрузка и строгая валидация данных

Проверяются уникальность \`hash\`, допустимые классы, диапазоны координат, дубли и пересечения. Дополнительно сверяются SHA-256 train/dev с manifest.


In [2]:
ENTITY_LABELS = ("ORG", "NAME", "GEO")
TAGS = ("O", "B-ORG", "I-ORG", "B-NAME", "I-NAME", "B-GEO", "I-GEO")
TAG_TO_ID = {tag: i for i, tag in enumerate(TAGS)}
ID_TO_TAG = {i: tag for tag, i in TAG_TO_ID.items()}

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def validate_record(raw: Any, source: str) -> dict[str, Any]:
    if not isinstance(raw, dict):
        raise ValueError(f"{source}: record must be an object")
    record_hash, text, entities = raw.get("hash"), raw.get("text"), raw.get("entities")
    if not isinstance(record_hash, str) or not record_hash:
        raise ValueError(f"{source}: invalid hash")
    if not isinstance(text, str):
        raise ValueError(f"{source}: text must be a string")
    if not isinstance(entities, list):
        raise ValueError(f"{source}: entities must be a list")

    clean, seen = [], set()
    for j, entity in enumerate(entities):
        if not isinstance(entity, dict):
            raise ValueError(f"{source}/entities[{j}]: must be an object")
        label, start, end = entity.get("label"), entity.get("start"), entity.get("end")
        if label not in ENTITY_LABELS:
            raise ValueError(f"{source}/entities[{j}]: invalid label {label!r}")
        if (
            not isinstance(start, int) or isinstance(start, bool)
            or not isinstance(end, int) or isinstance(end, bool)
            or not 0 <= start < end <= len(text)
        ):
            raise ValueError(f"{source}/entities[{j}]: invalid offsets")
        key = (label, start, end)
        if key in seen:
            raise ValueError(f"{source}/entities[{j}]: duplicate entity")
        seen.add(key)
        clean.append({"label": label, "start": start, "end": end})

    clean.sort(key=lambda e: (e["start"], e["end"], e["label"]))
    for left, right in zip(clean, clean[1:]):
        if right["start"] < left["end"]:
            raise ValueError(f"{source}: overlapping entities are unsupported")
    return {"hash": record_hash, "text": text, "entities": clean}

def read_jsonl(path: Path, limit: int | None = None) -> list[dict[str, Any]]:
    records, hashes = [], set()
    with path.open(encoding="utf-8") as stream:
        for line_no, line in enumerate(stream, 1):
            if not line.strip():
                raise ValueError(f"{path}:{line_no}: empty line")
            record = validate_record(json.loads(line), f"{path}:{line_no}")
            if record["hash"] in hashes:
                raise ValueError(f"{path}:{line_no}: duplicate hash")
            hashes.add(record["hash"])
            records.append(record)
            if limit is not None and len(records) >= limit:
                break
    if not records:
        raise ValueError(f"{path}: no records")
    return records

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
for split, path in (("train", TRAIN_PATH), ("dev", DEV_PATH)):
    expected = manifest["splits"][split]["sha256"]
    actual = sha256(path)
    if actual != expected:
        raise ValueError(f"SHA-256 mismatch for {path}: {actual} != {expected}")
print("SHA-256 train/dev совпадают с manifest.")

train_limit = CFG.fast_train_records if CFG.fast_run else None
dev_limit = CFG.fast_dev_records if CFG.fast_run else None
train_records = read_jsonl(TRAIN_PATH, train_limit)
dev_records = read_jsonl(DEV_PATH, dev_limit)
print(f"Train: {len(train_records):,}; dev: {len(dev_records):,}")


SHA-256 train/dev совпадают с manifest.
Train: 13,000; dev: 1,500


In [3]:
def split_stats(records: list[dict[str, Any]]) -> dict[str, Any]:
    labels = Counter(e["label"] for r in records for e in r["entities"])
    lengths = sorted(len(r["text"]) for r in records)
    return {
        "documents": len(records),
        "empty_documents": sum(not r["entities"] for r in records),
        "entities": sum(labels.values()),
        **{label: labels[label] for label in ENTITY_LABELS},
        "median_chars": lengths[len(lengths) // 2],
        "max_chars": lengths[-1],
    }

print("TRAIN", json.dumps(split_stats(train_records), ensure_ascii=False, indent=2))
print("DEV  ", json.dumps(split_stats(dev_records), ensure_ascii=False, indent=2))
print("\nПример exact spans:")
example = next(r for r in train_records if r["entities"])
print(example["text"][:350])
for entity in example["entities"][:8]:
    print(entity, "=>", repr(example["text"][entity["start"]:entity["end"]]))


TRAIN {
  "documents": 13000,
  "empty_documents": 2386,
  "entities": 66083,
  "ORG": 23420,
  "NAME": 21218,
  "GEO": 21445,
  "median_chars": 151,
  "max_chars": 22668
}
DEV   {
  "documents": 1500,
  "empty_documents": 275,
  "entities": 7698,
  "ORG": 2659,
  "NAME": 2319,
  "GEO": 2720,
  "median_chars": 171,
  "max_chars": 18975
}

Пример exact spans:
Тошкент вилоятининг Бўка-Бекобод йўлида ИИБ ходими томонидан уриб юборилиб, Дўстлик каналига ташлангани айтилган 43 ёшли фуқаронинг жасади 80 кун ўтиб топилди. Марҳумнинг яқинлари маълум қилишича, жасад 28-февраль куни соат 10:30 ларда Дўстлик каналидан, сув оқимининг қуйи қисмидан аниқланган. У қум остида қолиб кетган бўлган. Қариндошларининг айти
{'label': 'GEO', 'start': 0, 'end': 19} => 'Тошкент вилоятининг'
{'label': 'GEO', 'start': 20, 'end': 39} => 'Бўка-Бекобод йўлида'
{'label': 'ORG', 'start': 40, 'end': 43} => 'ИИБ'
{'label': 'GEO', 'start': 76, 'end': 92} => 'Дўстлик каналига'
{'label': 'GEO', 'start': 236, 'end': 253} =

## 3. Предобработка: tokenizer offsets → BIO

Почему именно так:

- тексты длиной до десятков тысяч символов превращаются в перекрывающиеся окна;
- fast tokenizer возвращает координаты каждого subword в исходной Python-строке;
- special tokens получают \`-100\` и не участвуют в loss;
- если сущность обрезана границей окна, её токены в этом окне маскируются. Полная копия сущности остаётся в соседнем окне;
- Unicode/NFKC, апострофы, регистр и пробелы не нормализуются: это сохраняет exact offsets.

Если граница gold попала внутрь одного tokenizer token, token classification не может восстановить её идеально. Следующая проверка показывает число таких случаев.


In [5]:
tokenizer = baseline_load_fast_tokenizer(CFG.model_name)
baseline_validate_window(tokenizer, CFG.max_length, CFG.stride)
from typing import Any

from tqdm.auto import tqdm

from baseline.common import tokenize_windows
"""def tokenize_windows(
    tokenizer: PreTrainedTokenizerBase,
    text: str,
    *,
    max_length: int,
    stride: int,
) -> list[tuple[ModelFeature, Offsets]]:

    encoded = tokenizer(
        text,
        truncation=False,
        max_length=max_length,
        stride=stride,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
    )
    input_chunks = encoded["input_ids"]
    offset_chunks = encoded["offset_mapping"]
    if input_chunks and isinstance(input_chunks[0], int):
        input_chunks = [input_chunks]
        offset_chunks = [offset_chunks]

    windows: list[tuple[ModelFeature, Offsets]] = []
    for chunk_index, offsets in enumerate(offset_chunks):
        feature: ModelFeature = {}
        for key in ("input_ids", "attention_mask"):
            if key not in encoded:
                continue
            values = encoded[key]
            feature[key] = values[chunk_index] if values and isinstance(values[0], list) else values
        windows.append((feature, [(int(start), int(end)) for start, end in offsets]))
    return windows """


def _token_to_dict(
    tokenizer,
    text: str,
    token_id: int,
    start: int,
    end: int,
) -> dict[str, Any]:
    return {
        "token": tokenizer.convert_ids_to_tokens(int(token_id)),
        "token_id": int(token_id),
        "text": text[start:end],
        "start": start,
        "end": end,
        "length": end - start,
    }


def render_all_token_boundaries(
    text,
    all_tokens,
    *,
    gold_start,
    gold_end,
    context_size=50,
):
    context_start = max(0, gold_start - context_size)
    context_end = min(len(text), gold_end + context_size)

    context_tokens = sorted(
        [
            token
            for token in all_tokens
            if (
                token["start"] < context_end
                and token["end"] > context_start
            )
        ],
        key=lambda token: (
            token["start"],
            token["end"],
        ),
    )

    result = []
    cursor = context_start

    for token in context_tokens:
        token_start = max(token["start"], context_start)
        token_end = min(token["end"], context_end)

        if token_end <= cursor:
            continue

        if token_start > cursor:
            # Обычно здесь находятся пробелы.
            result.append(text[cursor:token_start])

        result.append(
            "⟪" + text[token_start:token_end] + "⟫"
        )
        cursor = token_end

    if cursor < context_end:
        result.append(text[cursor:context_end])

    return "".join(result).replace("\n", "\\n")


def boundary_alignment_issues(
    records: list[dict[str, Any]],
    tokenizer,
    *,
    max_length: int,
    stride: int,
    context_size: int = 50,
) -> list[dict[str, Any]]:
    """
    Показывает токены, которые пересекают границы gold-сущностей.

    Проблемный большой токен:
      token.start < gold boundary < token.end

    То есть gold-граница находится внутри tokenizer-токена.
    """

    issues: list[dict[str, Any]] = []

    for record in tqdm(
        records,
        desc="Check boundary alignment",
        unit="doc",
    ):
        text = record["text"]

        # Собираем токены из всех sliding windows.
        # Одинаковые offsets из overlap-окон удаляем.
        tokens_by_offset: dict[
            tuple[int, int],
            dict[str, Any],
        ] = {}

        windows = tokenize_windows(
            tokenizer,
            text,
            max_length=max_length,
            stride=stride,
        )

        for feature, offsets in windows:
            input_ids = feature["input_ids"]

            for token_id, (start, end) in zip(
                input_ids,
                offsets,
                strict=True,
            ):
                # Special tokens обычно имеют offsets (0, 0).
                if start == end:
                    continue

                offset = (start, end)

                if offset not in tokens_by_offset:
                    tokens_by_offset[offset] = _token_to_dict(
                        tokenizer,
                        text,
                        token_id,
                        start,
                        end,
                    )

        tokens = sorted(
            tokens_by_offset.values(),
            key=lambda token: (
                token["start"],
                token["end"],
            ),
        )

        token_starts = {token["start"] for token in tokens}
        token_ends = {token["end"] for token in tokens}

        for entity in record["entities"]:
            gold_start = entity["start"]
            gold_end = entity["end"]

            start_aligned = gold_start in token_starts
            end_aligned = gold_end in token_ends

            if start_aligned and end_aligned:
                continue

            # Токены, внутри которых находится gold start.
            tokens_crossing_start = [
                token
                for token in tokens
                if token["start"] < gold_start < token["end"]
            ]

            # Токены, внутри которых находится gold end.
            tokens_crossing_end = [
                token
                for token in tokens
                if token["start"] < gold_end < token["end"]
            ]

            # Все токены, пересекающие gold-сущность.
            overlapping_tokens = [
                token
                for token in tokens
                if (
                    token["start"] < gold_end
                    and token["end"] > gold_start
                )
            ]

            # Уникальные проблемные большие токены.
            problem_tokens_by_offset = {
                (token["start"], token["end"]): token
                for token in (
                    tokens_crossing_start
                    + tokens_crossing_end
                )
            }

            problem_tokens = sorted(
                problem_tokens_by_offset.values(),
                key=lambda token: (
                    token["start"],
                    token["end"],
                ),
            )

            token_aligned_start = (
                min(token["start"] for token in overlapping_tokens)
                if overlapping_tokens
                else None
            )
            token_aligned_end = (
                max(token["end"] for token in overlapping_tokens)
                if overlapping_tokens
                else None
            )

            token_aligned_text = (
                text[token_aligned_start:token_aligned_end]
                if (
                    token_aligned_start is not None
                    and token_aligned_end is not None
                )
                else None
            )

            detailed_problem_tokens = []

            for token in problem_tokens:
                crosses_start = (
                    token["start"]
                    < gold_start
                    < token["end"]
                )
                crosses_end = (
                    token["start"]
                    < gold_end
                    < token["end"]
                )

                detailed_problem_tokens.append({
                    **token,
                    "crosses_gold_start": crosses_start,
                    "crosses_gold_end": crosses_end,

                    # Лишняя часть токена слева от gold.
                    "extra_before_gold": (
                        text[token["start"]:gold_start]
                        if crosses_start
                        else ""
                    ),

                    # Лишняя часть токена справа от gold.
                    "extra_after_gold": (
                        text[gold_end:token["end"]]
                        if crosses_end
                        else ""
                    ),
                })

            issues.append({
                "hash": record["hash"],
                "label": entity["label"],

                "gold_text": text[gold_start:gold_end],
                "gold_start": gold_start,
                "gold_end": gold_end,
                "gold_offsets": (gold_start, gold_end),

                "start_aligned": start_aligned,
                "end_aligned": end_aligned,

                # Именно токены, разрезающие gold-границы.
                "problem_tokens": detailed_problem_tokens,

                # Все токены, из которых tokenizer пытается
                # составить gold-сущность.
                "overlapping_tokens": overlapping_tokens,

                # Минимальный span из целых токенов.
                "token_aligned_text": token_aligned_text,
                "token_aligned_offsets": (
                    token_aligned_start,
                    token_aligned_end,
                ),

                # ⟪...⟫ обозначает проблемные tokenizer-токены.
                "tokenizer_context": render_all_token_boundaries(
                    text,
                    tokens,
                    gold_start=gold_start,
                    gold_end=gold_end,
                    context_size=context_size,
                ),
            })

    return issues

alignment_issues = boundary_alignment_issues(
    train_records + dev_records,
    tokenizer,
    max_length=CFG.max_length,
    stride=CFG.stride,
)

print(f"Найдено проблем: {len(alignment_issues):,}")


Check boundary alignment: 100%|██████████| 14500/14500 [00:06<00:00, 2307.94doc/s]

Найдено проблем: 17


In [6]:
# Используем готовый Dataset из baseline.common, а не локальную копию.
train_dataset = BaselineTokenizedNerDataset(
    train_records,
    tokenizer,
    max_length=CFG.max_length,
    stride=CFG.stride,
    description="Tokenize train via baseline",
)
collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)

generator = torch.Generator().manual_seed(CFG.seed)
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.train_batch_size,
    shuffle=True,
    collate_fn=collator,
    num_workers=CFG.num_workers,
    generator=generator,
    pin_memory=DEVICE.type == "cuda",
)
print(f"Train windows: {len(train_dataset):,}")
print("BIO alignment and sliding windows: baseline.common.TokenizedNerDataset")


Tokenize train via baseline: 100%|██████████| 13000/13000 [00:04<00:00, 2980.09doc/s]

Train windows: 13,000
BIO alignment and sliding windows: baseline.common.TokenizedNerDataset


## 4. Exact-span inference и метрики

Логиты одинаковых токенов из перекрывающихся окон усредняются. После BIO-decoding получаются исходные \`start/end\`. Итоговая метрика совпадает с условием: TP требует одновременного совпадения \`hash + label + start + end\`.


In [7]:
# Готовая сборка окон из baseline.predict.
dev_windows = baseline_build_windows(
    dev_records, tokenizer, max_length=CFG.max_length, stride=CFG.stride
)
print(f"Dev windows: {len(dev_windows):,}")

def decode_bio_tokens(tagged_tokens: list[tuple[int, int, str]]) -> list[dict[str, Any]]:
    entities, current = [], None
    for start, end, tag in tagged_tokens:
        if tag == "O":
            if current is not None:
                entities.append(current)
                current = None
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "B" or current is None or current["label"] != label:
            if current is not None:
                entities.append(current)
            current = {"label": label, "start": start, "end": end}
        else:
            current["end"] = max(current["end"], end)
    if current is not None:
        entities.append(current)
    # Защита формата: уникальные валидные spans в стабильном порядке.
    unique = {(e["label"], e["start"], e["end"]): e for e in entities}
    return sorted(unique.values(), key=lambda e: (e["start"], e["end"], e["label"]))

@torch.inference_mode()
def predict_records(
    model: torch.nn.Module,
    records: list[dict[str, Any]],
    windows,
) -> list[dict[str, Any]]:
    model.eval()
    aggregated: list[dict[tuple[int, int], tuple[torch.Tensor, int]]] = [
        {} for _ in records
    ]
    for batch_start in tqdm(
        range(0, len(windows), CFG.eval_batch_size),
        desc="Exact-span predict",
        unit="batch",
    ):
        batch_windows = windows[batch_start:batch_start + CFG.eval_batch_size]
        padded = tokenizer.pad(
            [feature for _, feature, _ in batch_windows],
            padding=True,
            return_tensors="pt",
        )
        padded = {key: value.to(DEVICE) for key, value in padded.items()}
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**padded).logits
        probabilities = torch.softmax(logits.float(), dim=-1).cpu()

        for row, (record_index, _, offsets) in enumerate(batch_windows):
            record_scores = aggregated[record_index]
            for token_index, (start, end) in enumerate(offsets):
                if start == end:
                    continue
                score = probabilities[row, token_index]
                if (start, end) in record_scores:
                    previous, count = record_scores[(start, end)]
                    record_scores[(start, end)] = (previous + score, count + 1)
                else:
                    record_scores[(start, end)] = (score.clone(), 1)

    output = []
    for record, scores in zip(records, aggregated):
        tagged = []
        for (start, end), (score_sum, count) in sorted(scores.items()):
            tag_id = int((score_sum / count).argmax().item())
            tagged.append((start, end, ID_TO_TAG[tag_id]))
        output.append({
            "hash": record["hash"],
            "entities": decode_bio_tokens(tagged),
        })
    return output

# Переопределяем публичную точку notebook: сам инференс выполняет baseline.predict.
@torch.inference_mode()
def predict_records(model, records, windows):
    scores = baseline_predict_token_scores(
        model, tokenizer, windows, len(records),
        batch_size=CFG.eval_batch_size, device=DEVICE,
    )
    return baseline_decode_records(records, scores, ID_TO_TAG)

def metric_values(tp: int, fp: int, fn: int) -> dict[str, float | int]:
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

def exact_span_metrics(
    gold_records: list[dict[str, Any]],
    predictions: list[dict[str, Any]],
) -> dict[str, Any]:
    gold_by_hash = {
        r["hash"]: {(e["label"], e["start"], e["end"]) for e in r["entities"]}
        for r in gold_records
    }
    pred_by_hash = {
        r["hash"]: {(e["label"], e["start"], e["end"]) for e in r["entities"]}
        for r in predictions
    }
    if set(gold_by_hash) != set(pred_by_hash):
        raise ValueError("Gold/prediction hash sets differ.")

    counts = {label: Counter(tp=0, fp=0, fn=0) for label in ENTITY_LABELS}
    for record_hash, gold in gold_by_hash.items():
        pred = pred_by_hash[record_hash]
        for label in ENTITY_LABELS:
            gold_label = {x for x in gold if x[0] == label}
            pred_label = {x for x in pred if x[0] == label}
            counts[label]["tp"] += len(gold_label & pred_label)
            counts[label]["fp"] += len(pred_label - gold_label)
            counts[label]["fn"] += len(gold_label - pred_label)

    by_label = {
        label: metric_values(c["tp"], c["fp"], c["fn"])
        for label, c in counts.items()
    }
    micro = metric_values(
        sum(c["tp"] for c in counts.values()),
        sum(c["fp"] for c in counts.values()),
        sum(c["fn"] for c in counts.values()),
    )
    macro = {
        key: sum(by_label[label][key] for label in ENTITY_LABELS) / len(ENTITY_LABELS)
        for key in ("precision", "recall", "f1")
    }
    return {"matching": "same hash and exact label/start/end", "by_label": by_label, "micro": micro, "macro": macro}

def print_metrics(metrics: dict[str, Any]) -> None:
    print(f"{'scope':<8} {'precision':>10} {'recall':>10} {'f1':>10} {'tp':>8} {'fp':>8} {'fn':>8}")
    print("-" * 72)
    for label in ENTITY_LABELS:
        x = metrics["by_label"][label]
        print(f"{label:<8} {x['precision']:10.4f} {x['recall']:10.4f} {x['f1']:10.4f} {x['tp']:8d} {x['fp']:8d} {x['fn']:8d}")
    x = metrics["micro"]
    print(f"{'micro':<8} {x['precision']:10.4f} {x['recall']:10.4f} {x['f1']:10.4f} {x['tp']:8d} {x['fp']:8d} {x['fn']:8d}")
    x = metrics["macro"]
    print(f"{'macro':<8} {x['precision']:10.4f} {x['recall']:10.4f} {x['f1']:10.4f}")


Tokenize: 100%|██████████| 1500/1500 [00:00<00:00, 2880.31doc/s]

Dev windows: 1,500


## 5. Модель и обучение

Checkpoint сохраняется после эпохи только при улучшении exact-span micro-F1. Это немного медленнее оценки по loss, зато напрямую оптимизирует критерий хакатона.


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    CFG.model_name,
    num_labels=len(TAGS),
    id2label=ID_TO_TAG,
    label2id=TAG_TO_ID,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)
updates_per_epoch = math.ceil(len(train_loader) / CFG.gradient_accumulation_steps)
total_updates = updates_per_epoch * CFG.epochs
warmup_steps = int(total_updates * CFG.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_updates,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def move_batch(batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

def train_one_epoch(epoch: int) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    weighted_loss, labeled_tokens = 0.0, 0
    progress = tqdm(train_loader, desc=f"Train epoch {epoch}", unit="batch")

    for batch_index, batch in enumerate(progress, 1):
        batch = move_batch(batch)
        group_start = ((batch_index - 1) // CFG.gradient_accumulation_steps) * CFG.gradient_accumulation_steps
        group_size = min(CFG.gradient_accumulation_steps, len(train_loader) - group_start)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            output = model(**batch)
            loss = output.loss

        scaler.scale(loss / group_size).backward()
        n_tokens = int((batch["labels"] != -100).sum().item())
        weighted_loss += float(loss.detach().item()) * n_tokens
        labeled_tokens += n_tokens

        should_step = batch_index % CFG.gradient_accumulation_steps == 0 or batch_index == len(train_loader)
        if should_step:
            scaler.unscale_(optimizer)
            clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        progress.set_postfix(loss=f"{loss.detach().item():.4f}")

    return weighted_loss / max(labeled_tokens, 1)

def save_checkpoint(metrics: dict[str, Any], history: list[dict[str, Any]]) -> None:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)
    baseline_config = {
        "schema_version": 1,
        "base_model": CFG.model_name,
        "tags": list(TAGS),
        "max_length": CFG.max_length,
        "stride": CFG.stride,
        "seed": CFG.seed,
    }
    (MODEL_DIR / "baseline_config.json").write_text(
        json.dumps(baseline_config, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    payload = {
        "schema_version": 1,
        "config": asdict(CFG),
        "tags": list(TAGS),
        "best_metrics": metrics,
        "history": history,
        "dataset_sha256": {
            "train": sha256(TRAIN_PATH),
            "dev": sha256(DEV_PATH),
        },
    }
    (MODEL_DIR / "ner_training_config.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

history = []
best_f1 = -1.0
best_predictions = None
started = time.time()

for epoch in range(1, CFG.epochs + 1):
    train_loss = baseline_train_epoch(
        model, train_loader, optimizer, scheduler, DEVICE,
        gradient_accumulation_steps=CFG.gradient_accumulation_steps,
        max_grad_norm=CFG.max_grad_norm,
    )
    predictions = predict_records(model, dev_records, dev_windows)
    metrics = exact_span_metrics(dev_records, predictions)
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "micro_precision": metrics["micro"]["precision"],
        "micro_recall": metrics["micro"]["recall"],
        "micro_f1": metrics["micro"]["f1"],
    }
    history.append(row)
    print(f"\nEpoch {epoch}: train_loss={train_loss:.6f}")
    print_metrics(metrics)

    if metrics["micro"]["f1"] > best_f1:
        best_f1 = metrics["micro"]["f1"]
        best_predictions = predictions
        save_checkpoint(metrics, history)
        print(f"Saved new best checkpoint: micro-F1={best_f1:.4f}")

print(f"\nDone in {(time.time() - started) / 60:.1f} min. Best micro-F1={best_f1:.4f}")
print(*history, sep="\n")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8040.46it/s]
[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Train:   2%|▏         | 37/1946 [00:12<08:54,  3.57batch/s, loss=1.8711]

## 6. Сохранение и официальный контрольный scorer

Формат JSONL полностью совместим с \`scripts/evaluate.py\`. Встроенный расчёт сверяется с официальным scorer; расхождение вызывает ошибку.


In [ ]:
if best_predictions is None:
    raise RuntimeError("Сначала выполните ячейку обучения.")

PREDICTIONS_PATH = ARTIFACT_DIR / "dev_predictions.jsonl"
METRICS_PATH = ARTIFACT_DIR / "dev_metrics.json"
HISTORY_PATH = ARTIFACT_DIR / "training_history.json"

with PREDICTIONS_PATH.open("w", encoding="utf-8") as stream:
    for record in best_predictions:
        stream.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n")

notebook_metrics = exact_span_metrics(dev_records, best_predictions)
METRICS_PATH.write_text(
    json.dumps(notebook_metrics, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
HISTORY_PATH.write_text(
    json.dumps(history, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

scripts_dir = str(PROJECT_DIR / "scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)
from evaluate import evaluate_files  # официальный scorer из комплекта

SCORER_GOLD_PATH = DEV_PATH
if CFG.fast_run:
    SCORER_GOLD_PATH = ARTIFACT_DIR / "dev_gold_subset.jsonl"
    with SCORER_GOLD_PATH.open("w", encoding="utf-8") as stream:
        for record in dev_records:
            stream.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n")
official_metrics = evaluate_files(SCORER_GOLD_PATH, PREDICTIONS_PATH)
assert abs(official_metrics["micro"]["f1"] - notebook_metrics["micro"]["f1"]) < 1e-12

print_metrics(notebook_metrics)
print("\nOfficial scorer agreement: OK")
print("Model:", MODEL_DIR)
print("Predictions:", PREDICTIONS_PATH)
print("Metrics:", METRICS_PATH)


## 7. Анализ ошибок

Для каждого FP ищется пересекающийся gold span. Ошибки делятся на:

- \`wrong_label\` — границы точные, класс неверный;
- \`boundary\` — есть пересечение, но границы отличаются;
- \`spurious\` — пересечения с gold нет;
- \`missed\` — gold span не найден.

Примеры помогают выбирать следующий эксперимент: улучшать декодирование границ, контекст класса или recall.


In [ ]:
def overlap(a: tuple[str, int, int], b: tuple[str, int, int]) -> int:
    return max(0, min(a[2], b[2]) - max(a[1], b[1]))

def collect_errors(
    gold_records: list[dict[str, Any]],
    predictions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    pred_by_hash = {r["hash"]: r["entities"] for r in predictions}
    errors = []
    for record in gold_records:
        text = record["text"]
        gold = {(e["label"], e["start"], e["end"]) for e in record["entities"]}
        pred = {(e["label"], e["start"], e["end"]) for e in pred_by_hash[record["hash"]]}

        for item in sorted(pred - gold):
            same_bounds = next((g for g in gold if g[1:] == item[1:]), None)
            overlaps = sorted(
                ((overlap(item, g), g) for g in gold if overlap(item, g) > 0),
                reverse=True,
            )
            kind = "wrong_label" if same_bounds else ("boundary" if overlaps else "spurious")
            reference = same_bounds or (overlaps[0][1] if overlaps else None)
            errors.append({
                "kind": kind,
                "hash": record["hash"],
                "pred": item,
                "gold": reference,
                "pred_text": text[item[1]:item[2]],
                "gold_text": text[reference[1]:reference[2]] if reference else None,
                "context": text[max(0, item[1] - 60):min(len(text), item[2] + 60)].replace("\n", " "),
            })

        for item in sorted(gold - pred):
            if not any(overlap(item, p) > 0 for p in pred):
                errors.append({
                    "kind": "missed",
                    "hash": record["hash"],
                    "pred": None,
                    "gold": item,
                    "pred_text": None,
                    "gold_text": text[item[1]:item[2]],
                    "context": text[max(0, item[1] - 60):min(len(text), item[2] + 60)].replace("\n", " "),
                })
    return errors

errors = collect_errors(dev_records, best_predictions)
print("Error counts:", Counter(e["kind"] for e in errors))
for kind in ("wrong_label", "boundary", "spurious", "missed"):
    print(f"\n--- {kind} ---")
    for item in [e for e in errors if e["kind"] == kind][:5]:
        print(json.dumps(item, ensure_ascii=False))


## 8. Итоги и следующие эксперименты

После полного прогона сохранены:

- \`artifacts/baseline_notebook/model/\` — лучший checkpoint и tokenizer, совместимый с \`baseline.predict\`;
- \`dev_predictions.jsonl\` — exact character spans;
- \`dev_metrics.json\` — per-class, micro и macro P/R/F1;
- \`training_history.json\` — динамика обучения.

Приоритетные улучшения:

1. сравнить baseline multilingual DistilBERT с \`xlm-roberta-base\` при одной схеме оценки;
2. подобрать learning rate, длину окна и stride;
3. считать F1 отдельно для Latin/Cyrillic/mixed, seen/unseen и длинных сущностей;
4. попробовать length-preserving аугментацию вариантов апострофов только на train;
5. для финального запуска зафиксировать окружение и не подбирать параметры по закрытому test.

Token accuracy сознательно не используется как главная метрика: класс \`O\` доминирует, а высокая точность токенов не гарантирует строгого совпадения границ сущности.
